# Geospatial-Temporal Data Visualization Lab

**Research question:** How did excess mortality vary across countries and over time from 2020 to mid-2022, and how do differences in population size, reporting coverage, and geographic distribution affect interpretation?

This project includes the dataset and generated outputs. All analysis code is embedded below in executable notebook cells. The code uses the CSV in this folder and writes figures/tables to `outputs/`.

## Run instructions
- Install packages: `python -m pip install -r requirements.txt`
- Keep `all_weekly_excess_deaths.csv` beside this notebook.
- Run every cell from top to bottom. The map downloads Natural Earth boundary data and needs internet.
- The stated period 2020-W01 through 2021-W52 contains **104 weeks**. ISO year 2020 has a W53, which is outside that specified end date.


## Code 1 — Load, validate, and build the balanced panel

In [ ]:
from pathlib import Path
import sys, json, warnings
import pandas as pd, numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

ROOT=Path.cwd()
OUT=ROOT/'outputs'; OUT.mkdir(exist_ok=True)
CSV=ROOT/'all_weekly_excess_deaths.csv'
if not CSV.exists(): raise FileNotFoundError(f'Missing dataset: {CSV}. Keep the CSV beside this notebook or update CSV above.')
df=pd.read_csv(CSV)
source_rows, source_cols = df.shape
df['start_date']=pd.to_datetime(df.start_date)
df['end_date']=pd.to_datetime(df.end_date)
df['week_start']=pd.to_datetime([f'{int(y)}-W{int(w):02d}-1' for y,w in zip(df.year,df.week)],format='%G-W%V-%u')
df['iso_week']=df.week_start.dt.strftime('%G-W%V')
df['month']=df.week_start.dt.to_period('M').dt.to_timestamp()
national_df=df[df.region.eq(df.country)].copy()
all_expected=[pd.Timestamp.fromisocalendar(y,w,1) for y in [2020,2021] for w in range(1, (53 if y==2020 else 52)+1)]
requested=[pd.Timestamp.fromisocalendar(2020, w,1) for w in range(1,53)]+[pd.Timestamp.fromisocalendar(2021,w,1) for w in range(1,53)]
# Assignment specifies the literal range 2020-W01 through 2021-W52 (104 weeks).
period_df=national_df[national_df.week_start.isin(requested)].copy()
counts=period_df.groupby('country').week_start.nunique()
complete=counts[counts.eq(len(requested))].index.tolist()
panel_df=period_df[period_df.country.isin(complete)].copy()
weekly=panel_df.groupby(['week_start','iso_week'],as_index=False).agg(excess_deaths=('excess_deaths','sum'))
top=weekly.nlargest(3,'excess_deaths').sort_values('week_start')
country_peaks=panel_df.loc[panel_df.groupby('country').excess_deaths.idxmax(),['country','week_start','excess_deaths']].sort_values('excess_deaths',ascending=False)
print(f'Loaded {source_rows:,} rows and {source_cols} source columns from {CSV.name}.')
print(f'National countries: {national_df.country.nunique()}; complete-panel countries: {len(complete)}; weeks per panel country: {len(requested)}.')

## Code 2 — Data profile and quality checks

In [ ]:
# Profile and checks
date_mismatch=int((df.start_date != df.week_start).sum())
misaligned_by_country=df.loc[df.start_date.ne(df.week_start)].groupby('country').size().to_dict()
profile={'rows':len(df),'cols':source_cols,'countries':df.country.nunique(),'regions':df.region.nunique(),'earliest_start':str(df.start_date.min().date()),'latest_start':str(df.start_date.max().date()),'earliest_iso_week':df.iso_week.min(),'latest_iso_week':df.iso_week.max(),'subnational_countries':sorted(df.loc[~df.region.eq(df.country),'country'].unique().tolist()),'missing':df.iloc[:,:source_cols].isna().sum().to_dict(),'duplicates_country_region_week':int(df.duplicated(['country','region','year','week']).sum()),'national_rows':len(national_df),'national_countries':national_df.country.nunique(),'iso2020_weeks':int(df.loc[df.year.eq(2020),'week'].nunique()),'source_start_date_vs_iso_monday_mismatches':date_mismatch,'mismatches_by_country':misaligned_by_country,'literal_period_weeks':len(requested),'panel_countries':len(complete),'panel_rows':len(panel_df),'excluded_incomplete':sorted(set(national_df.country)-set(complete)),'panel_countries_list':complete}
profile['deaths_identity_max_abs_error']=float((df.excess_deaths-(df.total_deaths-df.expected_deaths)).abs().max())
(OUT/'data_profile.json').write_text(json.dumps(profile,indent=2))
pd.DataFrame({'dtype':df.dtypes.astype(str),'missing_count':df.isna().sum(),'missing_pct':df.iloc[:,:source_cols].isna().mean()*100}).to_csv(OUT/'data_quality.csv')

sns.set_theme(style='whitegrid',context='notebook')
print(json.dumps(profile, indent=2, default=str))
print('Source column quality:')
print(pd.DataFrame({'dtype': df.iloc[:, :source_cols].dtypes.astype(str), 'missing_count': df.iloc[:, :source_cols].isna().sum(), 'missing_pct': df.iloc[:, :source_cols].isna().mean() * 100}).to_string())

## Task 1 — Load and explore the dataset

The file has 5,770 rows and 17 source columns, 46 countries and 46 regions. Source fields have no missing values and there are no duplicate country-region-week records. `excess_deaths` agrees with `total_deaths - expected_deaths` within a maximum floating point residual of 5.4e-11.

**Question 1: Which countries have subnational rows, and how did you find them?**

None. I compared `region` with `country` for every record; they match for all 5,770 rows, so this dataset contains only national-level observations.

**Question 2: Which 3–4 columns will you use most, and why?**

I use `country` to identify each series, `year` and `week` to align ISO reporting weeks, `excess_deaths` to show the weekly count difference, and `excess_deaths_pct_change` to compare the difference relative to expected deaths.
## Task 2 — Prepare the data

Dates are parsed, and ISO Mondays are reconstructed from `year` and `week`. The data contains all 46 national country series; the 2020-W01 to 2021-W52 balanced panel retains 43 countries and 4,472 country-week rows. Guatemala, Montenegro, and Tunisia do not have all 104 weeks. In 378 records from Britain (131), Canada (113), and South Africa (134), source `start_date` differs from the ISO Monday; the ISO year/week fields are used for alignment. Month aggregation assigns each week to the month containing its ISO Monday, even if the reporting week crosses a month boundary.

**Question 1: Why must the week be a date for a time axis?**

A date gives chronological sorting and proportional spacing, correctly distinguishes the same week number in different years, and handles year boundaries and ISO week 53.

**Question 2: What would go wrong if you skipped the national-row filter or the consistent-panel filter?**

Combining national and subnational totals could double-count deaths; this file has no subnational rows, but the equality check verifies that. Without a balanced panel, aggregate changes could reflect countries entering or leaving the series instead of changes in deaths, so countries with incomplete coverage are excluded rather than filling missing weeks with zero.


## Code 3 — Temporal charts

In [ ]:
# Weekly aggregate with top three peaks
fig,ax=plt.subplots(figsize=(12,5.6)); ax.plot(weekly.week_start,weekly.excess_deaths,color='#1d5d78',lw=1.8); ax.axhline(0,color='black',lw=.8)
offsets=[(-55,10),(0,24),(55,10)]
for i,(_,r) in enumerate(top.iterrows()):
 ax.scatter(r.week_start,r.excess_deaths,color='#bd3f42',zorder=5); ax.annotate(f"{r.iso_week}\n{r.excess_deaths:,.0f}",(r.week_start,r.excess_deaths),xytext=offsets[i],textcoords='offset points',ha='center',fontsize=8,arrowprops={'arrowstyle':'-','color':'#666','lw':.7})
ax.set_ylim(top=weekly.excess_deaths.max()*1.20)
ax.set(title=f"Weekly excess deaths, balanced panel ({len(complete)} countries)",xlabel='ISO week',ylabel='Excess deaths (deaths)'); fig.autofmt_xdate(); fig.tight_layout(); fig.savefig(OUT/'weekly_excess_panel.png',dpi=300); plt.close(fig)
# geographically representative percent small multiples
continent_map={'Australia':'Oceania','New Zealand':'Oceania','Japan':'Asia','South Korea':'Asia','South Africa':'Africa','Brazil':'Latin America','Chile':'Latin America','Canada':'North America','United States':'North America','United Kingdom':'Europe','Spain':'Europe','Italy':'Europe','Germany':'Europe','Mexico':'Latin America','Colombia':'Latin America','Argentina':'Latin America'}
available=set(national_df.country.unique()); chosen=[]
for c in ['Australia','Japan','South Africa','Brazil','Chile','Canada','United Kingdom','Spain','Mexico']:
 if c in available: chosen.append(c)
if len(chosen)<6:
 for c in national_df.country.unique():
  if c not in chosen: chosen.append(c)
  if len(chosen)==9: break
chosen=chosen[:9]
fig,axes=plt.subplots(3,3,figsize=(13,9),sharex=True,sharey=True); axes=axes.ravel()
for ax,c in zip(axes,chosen):
 q=panel_df[panel_df.country.eq(c)]; ax.plot(q.week_start,q.excess_deaths_pct_change*100,color='#287c8e',lw=1.05); ax.axhline(0,color='#555',lw=.6); ax.set_title(c,fontsize=10); ax.set_ylabel('% of expected' if ax in axes[::3] else '')
for ax in axes[len(chosen):]: ax.set_visible(False)
fig.suptitle('Weekly excess deaths as a share of expected deaths\nSelected countries; common scale',y=1.01); fig.tight_layout(); fig.savefig(OUT/'country_small_multiples.png',dpi=300,bbox_inches='tight'); plt.close(fig)
# single country choose Brazil if available (coverage adequate)
single='United States' if 'United States' in available else sorted(available)[0]
one=national_df[national_df.country.eq(single)].sort_values('week_start')
fig,ax=plt.subplots(figsize=(12,5)); ax.plot(one.week_start,one.excess_deaths,label='Excess deaths',lw=1.4); ax.plot(one.week_start,one.covid_deaths,label='Reported COVID-19 deaths',lw=1.2); ax.axhline(0,color='black',lw=.7); ax.set(title=f'{single}: weekly excess deaths and reported COVID-19 deaths',xlabel='ISO week',ylabel='Deaths per reporting week'); ax.legend(); fig.autofmt_xdate(); fig.tight_layout(); fig.savefig(OUT/'country_excess_vs_covid.png',dpi=300); plt.close(fig)

## Task 3 — Analyze temporal patterns

The balanced-panel weekly aggregate peaks at **2021-W01 (89,072), 2021-W02 (96,356), and 2021-W03 (98,673)**. The leading country-specific peaks in this panel are Mexico (2021-W03, 28,234), United States (2020-W52, 24,931), and South Africa (2021-W02, 15,629). Individual country peaks do not all occur together.

![Weekly balanced-panel excess deaths](outputs/weekly_excess_panel.png)

![Selected country small multiples](outputs/country_small_multiples.png)

![United States excess and reported COVID-19 deaths](outputs/country_excess_vs_covid.png)

**Question 1: When were the main waves, and did every country peak at the same time?**

The largest aggregate totals cluster in 2021-W01–W03. No: Spain and Italy peak in March 2020, the United States in 2020-W52, Mexico in 2021-W03, and Iran in 2021-W34. These are each series’ highest weekly observations, not a claim that each country had only one wave.

**Question 2: What events could explain the peaks?**

The early-2021 aggregate peak coincides in time with the documented winter 2020–21 high-transmission period in the United States; several later peaks coincide with periods commonly described as Delta waves. This timing is historical context only: the dataset and charts cannot establish that any event caused a particular mortality peak. Excess mortality may include direct and indirect pandemic effects and other mortality changes. [CDC MMWR](https://www.cdc.gov/mmwr/volumes/71/wr/mm7104e4.htm)

**Question 3: When is excess mortality higher than reported COVID-19 mortality in the selected country, and what might explain the gap?**

For the United States, excess deaths exceed reported COVID-19 deaths in 87 of 128 available weeks, across dates from 2020-02-24 to 2022-06-06; the largest weekly difference is about 10,240 deaths in 2021-W34. Possible explanations include incomplete reporting or attribution, indirect pandemic effects, baseline-model differences, and other mortality changes; the gap does not prove deaths directly caused by COVID-19.


## Code 4 — Choropleth map

In [ ]:
# choropleth using geopandas natural earth low-res GeoJSON from public raw dataset
mapped=False; unmatched=[]; mapped_n=0
try:
 import geopandas as gpd
 world=gpd.read_file('https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_110m_admin_0_countries.geojson')
 cum=panel_df.groupby('country',as_index=False).agg(cumulative_excess_deaths=('excess_deaths','sum'),expected=('expected_deaths','sum'),population=('population','last'),weeks=('week_start','nunique'))
 aliases={'United States':'United States of America','Russia':'Russia','South Korea':'South Korea','Czechia':'Czechia','Taiwan':'Taiwan','Iran':'Iran','Vietnam':'Vietnam','Bolivia':'Bolivia','Tanzania':'Tanzania','Syria':'Syria','Moldova':'Moldova','Venezuela':'Venezuela'}
 namecol='ADMIN' if 'ADMIN' in world.columns else 'name'
 world['join_name']=world[namecol]
 world['join_name']=world.join_name.replace({'United States of America':'United States','United Kingdom':'Britain','Czechia':'Czech Republic','Republic of Korea':'South Korea','Dem. Rep. Korea':'North Korea','Czech Republic':'Czechia','Russian Federation':'Russia','Iran (Islamic Republic of)':'Iran','Viet Nam':'Vietnam','Bolivia (Plurinational State of)':'Bolivia','United Republic of Tanzania':'Tanzania','Syrian Arab Republic':'Syria','Republic of Moldova':'Moldova','Venezuela (Bolivarian Republic of)':'Venezuela','Taiwan':'Taiwan','Congo':'Congo','Democratic Republic of the Congo':'Democratic Republic of the Congo'})
 joined=world.merge(cum,left_on='join_name',right_on='country',how='left'); unmatched=sorted(set(cum.country)-set(joined.loc[joined.country.notna(),'country'])); mapped_n=int(joined.cumulative_excess_deaths.notna().sum())
 vmax=max(abs(cum.cumulative_excess_deaths.min()),abs(cum.cumulative_excess_deaths.max()))
 fig,ax=plt.subplots(figsize=(14,7)); joined.plot(column='cumulative_excess_deaths',ax=ax,cmap='RdBu_r',vmin=-vmax,vmax=vmax,missing_kwds={'color':'#e5e5e5','label':'Not in dataset'},legend=True,legend_kwds={'label':'Cumulative excess deaths (2020-W01 to 2021-W52)','shrink':.65}); ax.set_axis_off(); ax.set_title('Cumulative excess deaths by country, 2020-W01–2021-W52\nGray: no matching country observation; Economist data'); fig.tight_layout(); fig.savefig(OUT/'cumulative_excess_map.png',dpi=300); plt.close(fig); mapped=True
 if (OUT/'map_limitation.txt').exists(): (OUT/'map_limitation.txt').unlink()
 cum['per_100k']=cum.cumulative_excess_deaths/cum.population*100000; cum.to_csv(OUT/'cumulative_country.csv',index=False)
except Exception as e:
 (OUT/'map_limitation.txt').write_text(f'Choropleth generation failed: {type(e).__name__}: {e}\n'); cum=panel_df.groupby('country',as_index=False).agg(cumulative_excess_deaths=('excess_deaths','sum'),expected=('expected_deaths','sum'),population=('population','last'),weeks=('week_start','nunique')); cum['per_100k']=cum.cumulative_excess_deaths/cum.population*100000; cum.to_csv(OUT/'cumulative_country.csv',index=False)

## Task 4 — Geographic distribution

The map covers 43 countries with complete 2020-W01–2021-W52 coverage. 42 match the Natural Earth boundaries; Malta is unmatched. The United States has the largest absolute cumulative excess (980,150), followed by Mexico (609,900), Iran (256,220), South Africa (232,509), and Peru (184,194). New Zealand is the only country with negative cumulative excess (−2,665; about −52 per 100,000). On a per-100,000 basis Bulgaria ranks highest (about 870), illustrating that population-adjusted rankings differ from absolute counts.

![Cumulative excess deaths choropleth](outputs/cumulative_excess_map.png)

**Question 1: Which countries and regions stand out, and which have negative excess?**

The United States leads in cumulative count, followed by Mexico, Iran, South Africa, and Peru; New Zealand is the only negative country. Per-capita rates change the ranking, with Bulgaria, Lithuania, and Romania highest, so absolute totals and population-adjusted burden tell different stories.

**Question 2: Which countries are missing from the dataset entirely, and how does that limit a global interpretation of the map?**

The dataset contains 46 countries, so countries outside its supplied list are absent; a complete named list requires selecting a world-country reference standard. Malta is in the dataset but unmatched to the map boundaries. The map describes reporting countries only and should not be read as a global estimate.


## Code 5 — Monthly heat map, summary, and exported tables

In [ ]:
# monthly weighted ratio, restricting country-month to full observed periods; Monday assignment
monthly=panel_df.groupby(['country','month'],as_index=False).agg(total_deaths=('total_deaths','sum'),expected_deaths=('expected_deaths','sum'),nweeks=('week_start','nunique'))
monthly['ratio_pct']=(monthly.total_deaths-monthly.expected_deaths)/monthly.expected_deaths*100
pivot=monthly.pivot(index='country',columns='month',values='ratio_pct'); order=cum.sort_values('cumulative_excess_deaths',ascending=False).country.tolist(); pivot=pivot.reindex(order)
fig,ax=plt.subplots(figsize=(16,max(8,len(pivot)*.24))); sns.heatmap(pivot,cmap='RdBu_r',center=0,vmin=-max(abs(np.nanmin(pivot.values)),abs(np.nanmax(pivot.values))),vmax=max(abs(np.nanmin(pivot.values)),abs(np.nanmax(pivot.values))),mask=pivot.isna(),ax=ax,cbar_kws={'label':'Monthly excess / expected deaths (%)'}); ax.set(title='Monthly excess mortality by country (week assigned to month of Monday)',xlabel='Month',ylabel='Country'); ax.set_xticklabels([pd.Timestamp(x).strftime('%b\n%Y') for x in pivot.columns],rotation=0,fontsize=8); fig.tight_layout(); fig.savefig(OUT/'monthly_country_heatmap.png',dpi=300); plt.close(fig)
monthly.to_csv(OUT/'monthly_country.csv',index=False)
# Save calculated result tables
top.to_csv(OUT/'top_weekly_peaks.csv',index=False); country_peaks.to_csv(OUT/'country_peak_weeks.csv',index=False)
pd.DataFrame({'country':chosen}).to_csv(OUT/'small_multiple_countries.csv',index=False)
print(json.dumps({'profile':profile,'top_peaks':top.to_dict('records'),'country':single,'map_available':mapped,'mapped_countries':mapped_n,'unmatched':unmatched,'largest_cumulative':cum.nlargest(5,'cumulative_excess_deaths')[['country','cumulative_excess_deaths','per_100k']].to_dict('records'),'lowest_cumulative':cum.nsmallest(5,'cumulative_excess_deaths')[['country','cumulative_excess_deaths','per_100k']].to_dict('records'),'country_peaks_top':country_peaks.head(8).to_dict('records'),'small_multiples':chosen},default=str,indent=2))
summary_rows = [
    ('Countries in source', int(df.country.nunique())),
    ('National-level countries', int(national_df.country.nunique())),
    ('Complete-panel countries', len(complete)),
    ('Weeks per complete country', len(requested)),
    ('Country-week records in panel', len(panel_df)),
    ('Top weekly aggregate peak', f"{top.sort_values('excess_deaths', ascending=False).iloc[0].iso_week}: {top.excess_deaths.max():,.0f}"),
    ('Largest cumulative excess', f"{cum.loc[cum.cumulative_excess_deaths.idxmax(), 'country']}: {cum.cumulative_excess_deaths.max():,.0f}"),
    ('Lowest cumulative excess', f"{cum.loc[cum.cumulative_excess_deaths.idxmin(), 'country']}: {cum.cumulative_excess_deaths.min():,.0f}"),
    ('Mapped / eligible countries', f"{mapped_n} / {len(cum)}"),
    ('Unmatched map countries', ', '.join(unmatched) if unmatched else 'None'),
]
summary_df = pd.DataFrame(summary_rows, columns=['Finding', 'Calculated result'])
summary_df.to_csv(OUT / 'results_summary.csv', index=False)
print(summary_df.to_string(index=False))

## Task 5 — Combine space and time

The heat map uses a weighted monthly ratio: `(sum(total_deaths) - sum(expected_deaths)) / sum(expected_deaths)`. It assigns each weekly record to the month of its ISO Monday, uses a diverging scale centered at zero, and masks missing values.

![Monthly excess mortality heat map](outputs/monthly_country_heatmap.png)

**Question: How did the high-excess areas move between waves, and why did you choose this visualization design over the other two?**

The largest values shift across countries and months: western European peaks appear early in 2020, Mexico and South Africa have large peaks around late 2020/early 2021, and Iran peaks later in 2021. A heat map makes every country-month visible in one static figure with one scale; it is easier to compare in a report than repeated maps and retains more detail than a few frames of an animation.

## Normalization, limitations, and references

Absolute counts tend to emphasize countries with larger populations. Excess deaths per 100,000 adjusts for population size, while excess as a percentage of expected deaths compares relative deviation from a baseline; these are related but different measures. Excess mortality is not interchangeable with reported COVID-19 mortality and is descriptive, not causal. The file covers 46 countries, not the whole world; three countries lack full selected-period coverage, and three countries have source-date/ISO alignment discrepancies.

Dataset: [The Economist COVID-19 excess-deaths tracker](https://github.com/TheEconomist/covid-19-excess-deaths-tracker), CC BY 4.0. Method: [WHO excess-mortality methods](https://www.who.int/publications/m/item/methods-for-estimating-the-excess-mortality-associatedwith-the-covid-19-pandemic/). Boundaries: [Natural Earth](https://github.com/nvkelso/natural-earth-vector).
